# UPSCALER BY VENOZK — Colab (T4 GPU)
Max-quality **2x AI upscale** (Real-ESRGAN x2plus) + **background removal** (BiRefNet / ISNet) for images and videos.

**Runtime → Change runtime type → T4 GPU**, then run the cells in order.

In [ ]:
#@title 1. Install (run once)
!nvidia-smi -L
!pip -q install spandrel "rembg[gpu]" onnxruntime-gpu opencv-python-headless tqdm
!wget -q -nc https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth

In [ ]:
#@title 2. Load models
BG_MODEL = "birefnet-general" #@param ["birefnet-general","birefnet-portrait","isnet-general-use","u2net_human_seg"]
import torch, cv2, numpy as np, subprocess, os
from spandrel import ModelLoader
from rembg import remove, new_session
from tqdm.auto import tqdm
from google.colab import files
dev = "cuda"
sr = ModelLoader().load_from_file("RealESRGAN_x2plus.pth").to(dev).eval().half()
session = new_session(BG_MODEL)

@torch.inference_mode()
def upscale(bgr, tile=512, pad=16):
    h, w = bgr.shape[:2]
    out = np.zeros((h*2, w*2, 3), np.uint8)
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    for y in range(0, h, tile):
        for x in range(0, w, tile):
            y0, x0 = max(y-pad, 0), max(x-pad, 0)
            y1, x1 = min(y+tile+pad, h), min(x+tile+pad, w)
            t = torch.from_numpy(rgb[y0:y1, x0:x1]).to(dev).permute(2,0,1)[None].half()/255
            o = sr(t).clamp(0,1)[0].permute(1,2,0).float().mul(255).byte().cpu().numpy()
            oy, ox = (y-y0)*2, (x-x0)*2
            th, tw = min(tile, h-y)*2, min(tile, w-x)*2
            out[y*2:y*2+th, x*2:x*2+tw] = o[oy:oy+th, ox:ox+tw]
    return cv2.cvtColor(out, cv2.COLOR_RGB2BGR)

def get_mask(bgr):
    m = np.array(remove(cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB), session=session, only_mask=True))
    return m[..., 0] if m.ndim == 3 else m
print("Models ready on", torch.cuda.get_device_name(0))

In [ ]:
#@title 3. Image: upload -> process -> download
MODE = "upscale + remove bg" #@param ["upscale","remove bg","upscale + remove bg"]
for name, data in files.upload().items():
    img = cv2.imdecode(np.frombuffer(data, np.uint8), cv2.IMREAD_COLOR)
    mask = get_mask(img) if "remove" in MODE else None      # mask on original (fast, sharp edges)
    if "upscale" in MODE: img = upscale(img)
    if mask is not None:
        mask = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_CUBIC)
        img = np.dstack([img, mask])
    out = os.path.splitext(name)[0] + "_venozk.png"
    cv2.imwrite(out, img); files.download(out)

In [ ]:
#@title 4. Video: upload -> process -> download (keeps audio)
UPSCALE = True       #@param {type:"boolean"}
REMOVE_BG = False    #@param {type:"boolean"}
BACKGROUND = "transparent (WebM)" #@param ["transparent (WebM)","green","black","white"]
COLORS = {"green":(0,255,0),"black":(0,0,0),"white":(255,255,255)}   # BGR

src = next(iter(files.upload()))
cap = cv2.VideoCapture(src)
fps, n = cap.get(cv2.CAP_PROP_FPS), int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
w, h = int(cap.get(3)), int(cap.get(4))
s = 2 if UPSCALE else 1
W, H = w*s, h*s
alpha = REMOVE_BG and BACKGROUND.startswith("transparent")
ext = "webm" if alpha else "mp4"
vcodec = ["-c:v","libvpx-vp9","-pix_fmt","yuva420p","-b:v","0","-crf","28","-c:a","libopus"] if alpha \
    else ["-c:v","libx264","-crf","16","-preset","medium","-pix_fmt","yuv420p","-c:a","aac"]
out = f"venozk_out.{ext}"
p = subprocess.Popen(["ffmpeg","-y","-loglevel","error","-f","rawvideo","-pix_fmt","bgra" if alpha else "bgr24",
    "-s",f"{W}x{H}","-r",str(fps),"-i","-","-i",src,"-map","0:v","-map","1:a?",*vcodec,"-shortest",out], stdin=subprocess.PIPE)
for _ in tqdm(range(n)):
    ok, f = cap.read()
    if not ok: break
    m = get_mask(f) if REMOVE_BG else None
    if UPSCALE: f = upscale(f)
    if m is not None:
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_LINEAR)
        if alpha: f = np.dstack([f, m])
        else:
            a = (m.astype(np.float32)/255)[..., None]
            f = (f*a + np.array(COLORS[BACKGROUND], np.float32)*(1-a)).astype(np.uint8)
    p.stdin.write(f.tobytes())
p.stdin.close(); p.wait()
files.download(out)